# 02 — Transform: union The Numbers mirrors (Phase 2/3)

**Input (read-only):** `data/raw/tidytuesday_movie_profit.csv` (2018 snapshot,
3,401 films, budgets ≤ $175M) + `data/raw/wwu_mirror_pages/all_*.html`
(2023 snapshot, top 1,000 films by budget). Provenance: `data/raw/README_provenance.md`.

**Output:** `data/interim/the_numbers_union.csv` — one film-level table of
nominal-dollar budgets and grosses, deduplicated on (normalized title, year).

**Union rule:** TidyTuesday rows are kept as the base (they carry
`distributor`, `mpaa_rating`, `genre`); WWU rows are added only when their
(normalized title, year) is absent from TidyTuesday — i.e., WWU patches the
>$175M blockbuster gap. A `source` column records where each row came from.

In [1]:
import re
from pathlib import Path

import pandas as pd

RAW = Path.cwd().parent / "data" / "raw"
INTERIM = Path.cwd().parent / "data" / "interim"
INTERIM.mkdir(exist_ok=True)

def norm_title(s):
    """lowercase, drop everything but letters/digits — join key for dedupe only"""
    return re.sub(r"[^a-z0-9]", "", s.lower())

## 1. Parse the WWU mirror pages (HTML → rows)

In [2]:
def parse_money(s):
    return int(re.sub(r"[^0-9]", "", s)) if re.search(r"\d", s) else None

def parse_page(path):
    html = path.read_text(encoding="latin-1")
    tbl = re.search(r"<table.*?</table>", html, re.S).group(0)
    rows = []
    for tr in re.findall(r"<tr.*?</tr>", tbl, re.S)[1:]:  # [0] is the header
        cells = [re.sub(r"<[^>]+>", "", c).replace("&nbsp;", " ").strip()
                 for c in re.findall(r"<t[dh].*?</t[dh]>", tr, re.S)]
        if len(cells) == 6:
            rows.append(cells)
    return rows

pages = sorted((RAW / "wwu_mirror_pages").glob("all_*.html"))
raw_rows = []
for p in pages:
    rows = parse_page(p)
    assert len(rows) == 100, f"{p.name}: expected 100 rows, got {len(rows)}"
    raw_rows += rows

wwu = pd.DataFrame(raw_rows, columns=["rank", "release_date", "movie",
                                       "production_budget", "domestic_gross", "worldwide_gross"])
for c in ["production_budget", "domestic_gross", "worldwide_gross"]:
    wwu[c] = wwu[c].map(parse_money)
wwu["release_date"] = pd.to_datetime(wwu["release_date"], format="%b %d, %Y", errors="coerce")
wwu = wwu.drop(columns="rank")

print(f"WWU: {len(wwu)} rows from {len(pages)} pages")
print("unparseable dates:", wwu["release_date"].isna().sum())
print("budget range: ${:,.0f} – ${:,.0f}".format(wwu["production_budget"].min(), wwu["production_budget"].max()))
wwu.head(3)

WWU: 1000 rows from 10 pages
unparseable dates: 2
budget range: $60,000,000 – $460,000,000


,release_date,movie,production_budget,domestic_gross,worldwide_gross
0,2022-12-09,Avatar: The Way of Water,460000000,647253626,2213579051
1,2019-04-23,Avengers: Endgame,400000000,858373000,2794731755
2,2011-05-20,Pirates of the Caribbean: On Stranger Tides,379000000,241071802,1045713802


## 2. Load the TidyTuesday snapshot

In [3]:
tt = pd.read_csv(RAW / "tidytuesday_movie_profit.csv", index_col=0)
tt["release_date"] = pd.to_datetime(tt["release_date"], format="%m/%d/%Y")
for c in ["production_budget", "domestic_gross", "worldwide_gross"]:
    tt[c] = tt[c].astype("int64")

print(f"TidyTuesday: {len(tt)} rows")
print("exact-duplicate (movie, year) pairs inside TT:",
      tt.assign(year=tt.release_date.dt.year).duplicated(["movie", "year"]).sum())
tt.head(3)

TidyTuesday: 3401 rows
exact-duplicate (movie, year) pairs inside TT: 1


,release_date,movie,production_budget,domestic_gross,worldwide_gross,distributor,mpaa_rating,genre
1,2007-06-22,Evan Almighty,175000000,100289690,174131329,Universal,PG,Comedy
2,1995-07-28,Waterworld,175000000,88246220,264246220,Universal,PG-13,Action
3,2017-05-12,King Arthur: Legend of the Sword,175000000,39175066,139950708,Warner Bros.,PG-13,Adventure


## 3. Union + dedupe on (normalized title, year)

In [4]:
tt["year"] = tt["release_date"].dt.year.astype("Int64")
wwu["year"] = wwu["release_date"].dt.year.astype("Int64")  # Int64, not float: NaT dates must not poison the key dtype
tt["key"] = tt["movie"].map(norm_title) + "_" + tt["year"].astype(str)
wwu["key"] = wwu["movie"].map(norm_title) + "_" + wwu["year"].astype(str)

# drop TT's single internal dupe pair (keep first)
tt = tt.drop_duplicates("key", keep="first")

overlap = set(tt["key"]) & set(wwu["key"])
wwu_new = wwu[~wwu["key"].isin(overlap)].copy()

tt["source"] = "tidytuesday_2018"
wwu_new["source"] = "wwu_mirror_2023"

union = pd.concat([tt, wwu_new], ignore_index=True)\
    .sort_values("production_budget", ascending=False)\
    .reset_index(drop=True)

assert len(overlap) > 0, "zero overlap is impossible given the budget ranges — key bug"

print(f"TT base: {len(tt)}  |  overlap: {len(overlap)}  |  WWU added: {len(wwu_new)}")
print(f"union: {len(union)} films")
print(f"in 1990-2015 window: {union['year'].between(1990, 2015).sum()}")
print("budget range now: ${:,.0f} – ${:,.0f}".format(union["production_budget"].min(), union["production_budget"].max()))

TT base: 3400  |  overlap: 588  |  WWU added: 412
union: 3812 films
in 1990-2015 window: 3024
budget range now: $250,000 – $460,000,000


## 4. Cross-mirror consistency check

For films present in BOTH snapshots, budgets should agree (same source table,
5 years apart). Big disagreements would mean our title+year key is matching
different films — check before trusting the dedupe.

In [5]:
both = tt.merge(wwu, on="key", suffixes=("_tt", "_wwu"))
budget_match = (both["production_budget_tt"] == both["production_budget_wwu"]).mean()
print(f"overlap films: {len(both)}; identical budget: {budget_match:.1%}")

diff = both[both["production_budget_tt"] != both["production_budget_wwu"]]
print(f"budget differs for {len(diff)} films (revisions between snapshots — eyeball below)")
diff[["movie_tt", "year_tt", "production_budget_tt", "production_budget_wwu"]].head(10)

overlap films: 588; identical budget: 99.3%
budget differs for 4 films (revisions between snapshots — eyeball below)


,movie_tt,year_tt,production_budget_tt,production_budget_wwu
55,The Meg,2018,150000000,178000000
196,Venom,2018,100000000,116000000
214,Starship Troopers,1997,100000000,105000000
553,Chicken Little,2005,60000000,150000000


## 5. Write the interim table

In [6]:
out = union[["movie", "release_date", "year", "production_budget",
             "domestic_gross", "worldwide_gross", "distributor",
             "mpaa_rating", "genre", "source"]]
out_path = INTERIM / "the_numbers_union.csv"
out.to_csv(out_path, index=False)
print(f"wrote {len(out):,} rows -> {out_path}")
print("\nmissingness (WWU rows have no distributor/rating/genre by construction):")
print(out.isna().sum())

wrote 3,812 rows -> /Users/trevorjmckee/Documents/Work/School/03_Summer_2026/DSC530/final_project/data/interim/the_numbers_union.csv

missingness (WWU rows have no distributor/rating/genre by construction):
movie                  0
release_date           2
year                   2
production_budget      0
domestic_gross         0
worldwide_gross        0
distributor          459
mpaa_rating          549
genre                412
source                 0
dtype: int64


## Findings log

- Union size / window size: see §3 output
- Cross-mirror budget agreement: see §4 — high agreement validates the
  title+year dedupe key that the Phase 5 fuzzy join will also rely on
- `domestic_gross == 0` rows exist (films unreleased/limited at snapshot time) —
  handle explicitly in Phase 5/6, don't silently treat as real zeros
- WWU-sourced rows lack distributor/mpaa_rating/genre (documented, expected)
- Grosses are NOMINAL; never mix with Pudding's inflation-adjusted `gross`